# Lecture 05 test: Transformer components

Ungraded classroom practice, in slide order. These small examples use Python's
standard library. No model, dataset, network connection, or GPU is needed.

- E01 follows the sinusoidal-frequency explanation.
- E02 follows the RoPE rotation explanation.
- E03 follows the multi-head mechanism.
- E04 follows residual addition, LayerNorm, and the order comparison.
- E05 follows the FFN mechanism and parameter-share explanation.

All angles in the code are radians. Positions start at zero.
- E06: count an encoder with explicit parameter assumptions.


## Practice E01 · build one position vector · 2 minutes

For width 4, the pair frequencies are 1 and 0.01. Predict `PE(0)` on paper,
then add `PE(1)` to the toy embedding `e = [1, 0, 1, 0]`.

Here `e` is the already-scaled token embedding: the original Transformer uses
`sqrt(d_model) * E[token]` before adding the position vector.

Run the cells below to check your answer. Try position 2 and compare the two pairs.

In [ ]:
from math import sin, cos, sqrt, pi, isclose


def sinusoidal(position, width):
    if width <= 0 or width % 2:
        raise ValueError('Use a positive even width.')
    pairs = [(sin(position / 10000 ** (2 * i / width)),
              cos(position / 10000 ** (2 * i / width)))
             for i in range(width // 2)]
    return [value for pair in pairs for value in pair]


for position in (0, 1, 2):
    print(position, [round(value, 6) for value in sinusoidal(position, 4)])

In [ ]:
e = [1, 0, 1, 0]  # Invented, already-scaled embedding.
x = [content + position for content, position in zip(e, sinusoidal(1, 4))]
assert sinusoidal(0, 4) == [0, 1, 0, 1]
expected = [1.8414709848078965, 0.5403023058681398,
            1.0099998333341667, 0.9999500004166653]
assert all(isclose(a, b, abs_tol=1e-12) for a, b in zip(x, expected))
print('Input:', [round(value, 3) for value in x])

## Practice E02 · shift both positions · 2 minutes

Keep the content vectors fixed at `q = k = (1, 0)` and use the toy frequency
30 degrees per token. Predict the dot products at positions `(m, n) = (3, 1)`
and `(6, 4)` before running the next cell. The query position is `m`; the key
position is `n`.

Then change the gap from 2 to 3. These are raw dot products, before attention
scaling, masking, and softmax. The toy frequency is chosen for clear geometry;
it differs from RoPE's standard first-pair frequency of 1 radian per token.

In [ ]:
def rotate_pair(pair, angle):
    x, y = pair
    return [cos(angle) * x - sin(angle) * y,
            sin(angle) * x + cos(angle) * y]


def dot(left, right):
    return sum(a * b for a, b in zip(left, right))


theta = pi / 6
for m, n in [(3, 1), (6, 4), (4, 1), (7, 4)]:
    score = dot(rotate_pair([1, 0], m * theta), rotate_pair([1, 0], n * theta))
    expected = 0.5 if m - n == 2 else 0.0
    assert isclose(score, expected, abs_tol=1e-12)
    print(f'm={m}, n={n}, gap={m-n}: {score:.3f}')

### Check the full multi-pair identity

For column vectors, `R(mθ).T @ R(nθ) = R((n-m)θ)`: the signed offset is
**n minus m**. Repeat the check with different query and key vectors and a
frequency per coordinate pair. A shared shift preserves their dot product
and each rotation preserves vector length.

This identity holds for fixed content vectors. It does not assert that an
entire model produces the same outputs when its prompt content changes.

In [ ]:
def rope(vector, position):
    width = len(vector)
    if width <= 0 or width % 2:
        raise ValueError('Use a positive even width.')
    pairs = [rotate_pair(vector[2*i:2*i+2], position / 10000 ** (2*i/width))
             for i in range(width // 2)]
    return [value for pair in pairs for value in pair]


q, k = [0.3, -0.8, 1.5, 0.2], [-0.2, 0.6, 0.9, -1.2]
m, n, shift = 3, 1, 7
score = dot(rope(q, m), rope(k, n))
assert isclose(dot(rope(q, m), rope(q, m)), dot(q, q), abs_tol=1e-12)
assert isclose(score, dot(q, rope(k, n-m)), abs_tol=1e-12)
assert isclose(score, dot(rope(q, m+shift), rope(k, n+shift)), abs_tol=1e-12)
print('Norm and relative-offset checks passed.')

## Practice E03 · multi-head dimensions · 1 minute

Use `d_model = 512`, `heads = 8`, and `tokens = 4`. Assume equal key and value
widths, each `d_model / heads`, as in the original Transformer's base model.

Before running the next cell, predict:

1. The width of one head.
2. The shape after concatenating all heads.
3. The shape of the output projection and final output.

The demonstration supplies invented head outputs so we can inspect the
concatenation axis. It does not simulate learned attention weights. Each
number labels a head and token row; no trained model or dataset is used.

In [ ]:
n, d_model, h = 4, 512, 8
d_head = d_model // h
assert d_head == 64
# H[head][token][feature]: precomputed toy head outputs.
H = [[[100 * head + token] * d_head for token in range(n)] for head in range(h)]
# Join features for each token, keeping its row unchanged.
joined = [[value for head in H for value in head[token]] for token in range(n)]
assert (len(joined), len(joined[0])) == (4, 512)
for token in range(n):
    for head in range(h):
        assert joined[token][head * d_head:(head + 1) * d_head] == H[head][token]
projection_shape = (h * d_head, d_model)
output_shape = (n, projection_shape[1])
assert projection_shape == (512, 512) and output_shape == (4, 512)
print('One head:', (n, d_head))
print('Concatenated:', (len(joined), len(joined[0])))
print('W_O:', projection_shape, 'Output:', output_shape)

Each head has width **64**. The head outputs are **4 × 64**; concatenating their
features gives **4 × 512**. `W_O` is **512 × 512**, so the output remains **4 × 512**.
The number of token positions stays four.

Each head's query, key, and value projection is **512 × 64**. A head can use
all 512 input coordinates through its own learned projection; it is not
restricted to a fixed 64-coordinate slice of the raw input.

Reading: [Attention Is All You Need, section 3.2.2](https://arxiv.org/html/1706.03762v7#S3.S2).

## Practice E04 · normalize, then add the update · 2 minutes

For one token, use `x = [1, 3]`, `gamma = [1, 1]`, `beta = [0, 0]`, and
`eps = 1e-5`. The pre-LN update is given: `F(LN(x)) = [2, -1]`.

Before running the code, calculate the mean, population variance, normalized
vector, and residual output `y = x + F(LN(x))`. Can another token change
`LN(x)`? Distinguish the normalization from an attention update, which can
read allowed earlier tokens.

The variance divides by the feature count. Epsilon goes **inside** the square
root. Each token's feature vector has its own statistics. Learned scale and
bias are shared across token positions within one LayerNorm module.


In [ ]:
def layer_norm_features(vector, gamma, beta, eps=1e-5):
    """Normalize one token's features using population variance."""
    if not vector or not (len(vector) == len(gamma) == len(beta)):
        raise ValueError('Use matching nonempty feature vectors.')
    if eps <= 0:
        raise ValueError('Use a positive epsilon.')
    mean = sum(vector) / len(vector)
    variance = sum((value - mean) ** 2 for value in vector) / len(vector)
    return [scale * (value - mean) / sqrt(variance + eps) + bias
            for value, scale, bias in zip(vector, gamma, beta)]


x, gamma, beta = [1, 3], [1, 1], [0, 0]
mean = sum(x) / len(x)
variance = sum((value - mean) ** 2 for value in x) / len(x)
normalized = layer_norm_features(x, gamma, beta)
update = [2, -1]  # Given F(LN(x)); no model is trained here.
y = [value + change for value, change in zip(x, update)]
assert mean == 2 and variance == 1
expected = [-1 / sqrt(1.00001), 1 / sqrt(1.00001)]
assert all(isclose(a, b, abs_tol=1e-12) for a, b in zip(normalized, expected))
assert y == [3, 2]
print('Mean:', mean, 'Population variance:', variance)
print('LN(x):', [round(value, 6) for value in normalized], 'Residual output:', y)


### Check the axis and constant-vector case

The nested list below has shape `(B, T, d) = (1, 2, 2)`. Normalize its two token
rows separately, then perturb only the second row. Predict which output row
can change. A constant input has zero centered features, so LayerNorm returns
`beta`; epsilon keeps the denominator nonzero.


In [ ]:
def normalize_tokens(batch, gamma, beta):
    return [[layer_norm_features(token, gamma, beta) for token in sequence]
            for sequence in batch]


batch = [[[1, 3], [10, 14]]]  # Invented batch, token, and feature axes.
before = normalize_tokens(batch, gamma, beta)
batch[0][1] = [1000, -1000]  # Change only the other token.
after = normalize_tokens(batch, gamma, beta)
assert before[0][0] == after[0][0]
assert before[0][1] != after[0][1]
assert layer_norm_features([5, 5], [2, 3], [0.5, -0.5]) == [0.5, -0.5]
scaled = layer_norm_features(x, [2, 3], [0.5, -0.5])
assert all(isclose(a, b, abs_tol=1e-12) for a, b in zip(
    scaled, [2 * expected[0] + 0.5, 3 * expected[1] - 0.5]))
print('Token independence, constant-input, and learned-affine checks passed.')


The normalized vector is approximately **(-0.999995, 0.999995)** and the
pre-LN output is **(3, 2)**. Add the update to the original `x`.

Changing the other token leaves this token's LayerNorm unchanged. This check
isolates normalization; attention can still mix information from allowed
positions. LayerNorm recomputes statistics from each input in both training
and evaluation, and uses no batch running averages.

For a zero update, a pre-LN sublayer passes `x` through unchanged; a post-LN
sublayer would return `LN(x)`. After learned scale and bias, a LayerNorm output
need not have mean zero or variance one.

Readings: [Ba et al., §3](https://arxiv.org/html/1607.06450v1#S3),
[PyTorch LayerNorm formula](https://github.com/pytorch/pytorch/blob/v2.8.0/torch/nn/modules/normalization.py#L88),
and [Xiong et al., §§2–4](https://proceedings.mlr.press/v119/xiong20b.html).


## Practice E05 · apply the FFN to one token · 2 minutes

Use the invented row vector `x = [1, -2]`, zero biases, and these matrices:

```text
W1 = [[1, -1,  2],      W2 = [[1,  0],
      [0,  1, -1]]            [0,  1],
                              [1, -1]]
```

Compute `x W1`, apply ReLU coordinate by coordinate, and multiply by `W2`.
Can changing another token's supplied input change this token's FFN output?
The widths here are **2 → 3 → 2**. The diagram's 4× expansion is a design choice.


In [ ]:
def affine_features(vector, weights, bias):
    """Multiply a row vector by a weight matrix and add the bias."""
    return [sum(vector[i] * weights[i][j] for i in range(len(vector))) + bias[j]
            for j in range(len(bias))]


ffn_w1 = [[1, -1, 2], [0, 1, -1]]
ffn_w2 = [[1, 0], [0, 1], [1, -1]]
ffn_b1, ffn_b2 = [0, 0, 0], [0, 0]


def token_ffn(vector):
    expanded = affine_features(vector, ffn_w1, ffn_b1)
    activated = [max(0, value) for value in expanded]
    return affine_features(activated, ffn_w2, ffn_b2)


ffn_x = [1, -2]
expanded = affine_features(ffn_x, ffn_w1, ffn_b1)
activated = [max(0, value) for value in expanded]
ffn_output = token_ffn(ffn_x)
assert expanded == [1, -3, 4]
assert activated == [1, 0, 4]
assert ffn_output == [5, -4]
print('Expanded:', expanded, 'After ReLU:', activated, 'FFN output:', ffn_output)


### Token independence and nonlinearity

Apply the same FFN to two token vectors. Change only the second input and
check which output changes. This isolates the FFN; attention earlier in the
block can still change the contextual vector supplied to it.

Also compare `FFN(x) + FFN(-x)` with `FFN(0)`. With zero biases, a linear map
would make these equal. Predict what ReLU changes.


In [ ]:
ffn_inputs = [[1, -2], [0, 1]]
outputs_before = [token_ffn(token) for token in ffn_inputs]
changed_inputs = [ffn_inputs[0], [1000, -1000]]
outputs_after = [token_ffn(token) for token in changed_inputs]
assert outputs_before == [[5, -4], [0, 1]]
assert outputs_before[0] == outputs_after[0]
assert outputs_before[1] != outputs_after[1]
assert (len(outputs_before), len(outputs_before[0])) == (2, 2)

opposite_output = token_ffn([-value for value in ffn_x])
assert opposite_output == [0, 3]
assert [a + b for a, b in zip(ffn_output, opposite_output)] != token_ffn([0, 0])
print('Shared parameters, preserved token rows, and nonlinearity checks passed.')


### Check the two-thirds parameter share

For standard multi-head attention, the aggregate Q, K, V, and output matrices
contain `4 * d**2` weights. The two FFN matrices contain `2 * d * d_ff` weights.
At `d_ff = 4*d`, the FFN has two thirds of these **block matrix parameters**.
This count excludes embeddings, biases, normalization, and any extra attention
sublayers. Parameter count is one description of capacity, not a performance score.


In [ ]:
d, d_ff = 512, 2048
attention_parameters = 4 * d * d
ffn_parameters = 2 * d * d_ff
ffn_share = ffn_parameters / (attention_parameters + ffn_parameters)
assert attention_parameters == 1048576
assert ffn_parameters == 2097152
assert isclose(ffn_share, 2 / 3)
# A narrower hidden layer changes the share even before adding embeddings.
narrow_ffn = 2 * d * (2 * d)
assert narrow_ffn / (attention_parameters + narrow_ffn) == 0.5
print('Attention matrix parameters:', attention_parameters)
print('FFN matrix parameters:', ffn_parameters, 'Block share:', round(ffn_share, 4))


The intermediate vectors are **(1, -3, 4)** and **(1, 0, 4)**. The FFN output
is **(5, -4)**, before residual addition. All token positions use the same weights,
but each supplied vector is processed separately.

The nonlinear activation matters: `FFN(x) + FFN(-x) = [5, -1]`, whereas
`FFN([0, 0]) = [0, 0]`. Attention also has a nonlinear softmax, so this example
does not imply that removing the FFN makes an entire Transformer linear.

At widths 512 and 2048, the FFN matrices have 2,097,152 parameters and the
attention matrices have 1,048,576. The FFN share is 2/3 of their total.
Whole-model accounting must also include the other components and architecture.

Readings: [Vaswani et al., §3.3](https://arxiv.org/html/1706.03762v7#S3.S3)
and [GPT-2's GELU MLP](https://github.com/openai/gpt-2/blob/master/src/model.py).


## Practice E06 · count an encoder · 3 minutes

Use 12 layers, model width 768, 12 heads, vocabulary size 30,522, and FFN width `4 * width`.
Count token embeddings, all linear weights and biases, and two affine LayerNorms per layer.
Use sinusoidal positions. Exclude segment embeddings, embedding normalization, pooling,
extra final normalization, and prediction heads. This is the instructor's encoder-only
exercise, not the paper's base/big model or an exact BERT parameter count.

Predict the component counts before running the explicit shape tally below.
Splitting a fixed model width into more heads does not create another full-width matrix per head.


In [ ]:
from math import prod


def encoder_parameter_counts(layers, width, heads, vocabulary):
    """Count the specified encoder, including linear biases and LayerNorm."""
    if any(type(value) is not int or value <= 0 for value in (layers, width, heads, vocabulary)):
        raise ValueError("dimensions and layer count must be positive integers")
    if width % heads:
        raise ValueError("model width must be divisible by heads")
    head_width = width // heads
    shapes = {
        "query.weight": (heads * head_width, width), "query.bias": (heads * head_width,),
        "key.weight": (heads * head_width, width), "key.bias": (heads * head_width,),
        "value.weight": (heads * head_width, width), "value.bias": (heads * head_width,),
        "output.weight": (width, heads * head_width), "output.bias": (width,),
        "ffn_expand.weight": (4 * width, width), "ffn_expand.bias": (4 * width,),
        "ffn_contract.weight": (width, 4 * width), "ffn_contract.bias": (width,),
        "norm1.scale": (width,), "norm1.shift": (width,),
        "norm2.scale": (width,), "norm2.shift": (width,),
    }
    one_layer = sum(prod(shape) for shape in shapes.values())
    embeddings = prod((vocabulary, width))
    return {"embeddings": embeddings, "one_layer": one_layer,
            "stack": layers * one_layer, "total": embeddings + layers * one_layer}


parameter_counts = encoder_parameter_counts(12, 768, 12, 30522)
assert parameter_counts == {"embeddings": 23_440_896, "one_layer": 7_087_872,
                            "stack": 85_054_464, "total": 108_495_360}
assert parameter_counts["total"] == 30522 * 768 + 12 * (12 * 768**2 + 13 * 768)
assert encoder_parameter_counts(12, 768, 1, 30522) == parameter_counts
print(parameter_counts)


**Check:** embeddings contain **23,440,896** parameters; one layer contains **7,087,872**;
the 12-layer stack contains **85,054,464**. The total is **108,495,360**, about **108.50M**.

For one layer: attention has `4*d*d + 4*d`, the FFN has `8*d*d + 5*d`,
and two LayerNorms have `4*d`. Add `V*d` once for the token table.
Residual additions, dropout, softmax, and fixed sinusoidal positions have no learned parameters.
Different choices about positional embeddings or output heads produce different totals.


## Read a paper's training-cost comparison

[Attention Is All You Need, Table 2](https://arxiv.org/html/1706.03762v7#S6.S1)
reports estimated training FLOPs. For English–French, compare Transformer big
(`2.3e19`) with single ConvS2S (`1.5e20`) and its ensemble (`1.2e21`).
Calculate both ratios. These are neither measured inference-speed ratios nor
controlled wall-clock comparisons on identical hardware.


In [ ]:
single_cost_ratio = 1.5e20 / 2.3e19
ensemble_cost_ratio = 1.2e21 / 2.3e19
assert isclose(single_cost_ratio, 6.521739130434782)
assert isclose(ensemble_cost_ratio, 52.17391304347826)
print(f"Single ConvS2S / Transformer big: {single_cost_ratio:.2f}")
print(f"ConvS2S ensemble / Transformer big: {ensemble_cost_ratio:.2f}")


## Readings

- [Attention Is All You Need, sections 3.4–3.5](https://arxiv.org/html/1706.03762v7#S3.S5).
- [RoFormer, sections 3.2.1–3.2.2](https://arxiv.org/html/2104.09864v5).

The notebook and diagrams use original, invented numerical examples.
- [Attention Is All You Need, sections 5–6](https://arxiv.org/html/1706.03762v7#S5): experimental setup, translation, ablations, and parsing.
- [Reformer (2020)](https://arxiv.org/abs/2001.04451) and [Linformer (2020)](https://arxiv.org/abs/2006.04768): later efficiency methods.
- [Narang et al. (2021)](https://aclanthology.org/2021.emnlp-main.465/): transfer of architecture modifications.
